# 🔬 Lab: ChromaDB + RAG — Busqueda semantica para una fintech

**Workshop: Mas alla de SQL** | ITESM — Inteligencia de Negocios

### Escenario: MicroPréstamos MX 🇲🇽

Eres parte del equipo de datos de **MicroPréstamos MX**, una startup fintech mexicana que otorga micro-créditos. La empresa tiene datos **no estructurados** dispersos en múltiples canales:

- 💬 Conversaciones del **chatbot** en la app
- 📧 **Emails** de quejas de clientes
- 📞 **Transcripts** de llamadas del depto. de cobranza
- 📱 **Comentarios** en redes sociales (Facebook, Twitter)
- 📋 **Documentos internos** (políticas, procesos)

Tu reto: usar **ChromaDB** para hacer búsqueda semántica sobre estos datos y construir un mini-RAG que ayude al equipo a encontrar información rápido.

También cargarás un **dataset relacional** (CSV) para comparar datos estructurados vs no estructurados.

---

En este notebook vas a:
1. Crear una base de datos vectorial con datos reales de una fintech
2. Hacer búsquedas semánticas por significado
3. Visualizar los embeddings en 2D con PCA y t-SNE
4. Comparar datos estructurados (CSV) vs no estructurados (ChromaDB)
5. Construir un mini-RAG

---

**Setup**: Este notebook está diseñado para Google Colab. No necesitas instalar nada en tu computadora.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/mas-alla-de-sql/blob/main/labs/lab-chroma-rag.ipynb)

In [ ]:
# Instalar dependencias (solo necesario en Colab)
!pip install -q chromadb scikit-learn matplotlib numpy pandas

## Seccion 1 — Los datos de MicroPréstamos MX

La startup tiene **25 documentos no estructurados** de distintos canales. Cada documento tiene metadata indicando su **fuente** (canal de origen).

Estos son datos que SQL **no puede buscar por significado** — solo por palabras exactas (`LIKE '%palabra%'`).

Vamos a cargarlos en ChromaDB para hacer busqueda semantica.

In [ ]:
import chromadb

# Crear cliente en memoria (no necesita servidor)
client = chromadb.Client()

# Crear coleccion — ChromaDB genera los embeddings automaticamente
collection = client.create_collection("microprestamos_mx")

# 25 documentos no estructurados de MicroPréstamos MX
documents = [
    # --- Chatbot (app de la empresa) ---
    "Hola, quiero saber si puedo pedir un prestamo de 5000 pesos, es para pagar una emergencia medica de mi mama",
    "Ya hice mi pago pero no se refleja en la app, pague ayer en OXXO con el codigo de barras",
    "Me pueden explicar como funciona la tasa de interes? Vi que dicen 2% mensual pero no entiendo si es fija o variable",
    "Buenas tardes, mi solicitud lleva 3 dias en revision y necesito el dinero urgente para la renta",
    "Quiero aumentar mi linea de credito, ya llevo 6 meses pagando puntual todos mis prestamos",

    # --- Emails de quejas ---
    "Asunto: COBRO INDEBIDO. Me estan cobrando un prestamo que ya liquide hace 2 meses. Tengo mi comprobante de pago. Si no corrigen voy a ir a CONDUSEF.",
    "Asunto: Acoso de cobranza. Me llaman 5 veces al dia a mi trabajo y a mis referencias personales. Esto es ilegal segun la ley de proteccion al consumidor.",
    "Asunto: Error en mi historial crediticio. Aparezco como moroso en Buro de Credito pero mis pagos estan al corriente. Necesito que corrijan esto URGENTE.",
    "Asunto: Tasa abusiva. Cuando contrate me dijeron 1.5% mensual y ahora me cobran 3.8%. Esto es fraude y lo voy a reportar en redes sociales.",
    "Asunto: No puedo acceder a mi cuenta. Llevo una semana sin poder entrar a la app, ya reinstale y nada. Mientras tanto se me paso la fecha de pago.",

    # --- Cobranza (respuestas del departamento) ---
    "Estimado cliente, le recordamos que su pago de $2,350 vencio hace 15 dias. Puede pagar en cualquier OXXO, 7-Eleven o transferencia bancaria. Evite cargos moratorios.",
    "Hemos intentado contactarlo multiples veces sin exito. Su cuenta sera turnada a despacho externo de cobranza si no regulariza su situacion en 5 dias habiles.",
    "Le informamos que su caso fue escalado al area juridica debido a 90 dias de morosidad. Aun puede negociar un convenio de pago. Contactenos al 800-MICRO-MX.",
    "Confirmamos recepcion de su pago parcial de $500. Su saldo pendiente es $4,200. Le recomendamos liquidar antes del dia 30 para evitar intereses adicionales.",

    # --- Transcripts de llamadas ---
    "Transcript llamada #4521: Cliente menciona que perdio su empleo hace 2 meses y no puede pagar. Solicita reestructura de deuda. Agente ofrece plan de 12 meses con reduccion de tasa.",
    "Transcript llamada #4587: Cliente muy molesto, dice que le marcaron a su mama y le dijeron que tiene una deuda. Cliente amenaza con demanda por violacion a datos personales.",
    "Transcript llamada #4602: Cliente pregunta si puede pagar en parcialidades mas chicas. Actualmente paga $800 quincenal, quiere bajar a $400. Se le explica que se extiende el plazo.",
    "Transcript llamada #4615: Cliente solicita su estado de cuenta completo. Menciona que va a tramitar un credito hipotecario y necesita comprobar que no tiene adeudos.",

    # --- Redes sociales (Facebook, Twitter) ---
    "PESIMA experiencia con MicroPrestamos, me cobraron comisiones que nunca me explicaron. NO LOS RECOMIENDO. #fraude #fintech",
    "Yo si recomiendo MicroPrestamos, me salvaron cuando necesite lana para el taller de mi carro. El proceso fue rapido y por app. 👍",
    "Alguien sabe como contactar a MicroPrestamos? Su telefono no sirve y en la app no me contestan el chat desde hace 3 dias 😤",
    "@MicroPrestamosMX ya paguen lo que deben a sus inversionistas!! Llevo 4 meses sin recibir mis rendimientos. Esto es un ponzi?",
    "Acabo de liquidar mi prestamo con MicroPrestamos. Cero quejas, todo transparente. Gracias! 🙌 #fintech #mexico",

    # --- Documentos internos ---
    "POLITICA DE COBRANZA v3.2: El contacto con clientes morosos debe limitarse a 2 llamadas por dia en horario de 8am a 8pm. Queda prohibido contactar referencias personales antes de los 30 dias de mora.",
    "PROCESO DE ORIGINACION: Toda solicitud de prestamo debe pasar por score crediticio (Buro + modelo interno), validacion de identidad (INE + selfie), y comprobante de ingresos. Tiempo maximo de respuesta: 24 horas.",
    "REPORTE MENSUAL RIESGOS: La cartera vencida aumento 2.3% en el ultimo trimestre. Los principales factores son desempleo (40%), sobreendeudamiento (35%) y emergencias medicas (25%). Se recomienda ajustar el modelo de scoring.",
]

# Fuente de cada documento
sources = [
    "chatbot", "chatbot", "chatbot", "chatbot", "chatbot",
    "queja", "queja", "queja", "queja", "queja",
    "cobranza", "cobranza", "cobranza", "cobranza",
    "llamada", "llamada", "llamada", "llamada",
    "redes_sociales", "redes_sociales", "redes_sociales", "redes_sociales", "redes_sociales",
    "interno", "interno", "interno",
]

# Insertar documentos con metadata
collection.add(
    documents=documents,
    metadatas=[{"fuente": src} for src in sources],
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print(f"✅ Coleccion creada con {collection.count()} documentos de MicroPréstamos MX")
print(f"\nFuentes: {', '.join(sorted(set(sources)))}")

## Seccion 2 — Busquedas semanticas

Imagina que el equipo legal necesita encontrar **todos los casos donde clientes amenazaron con acciones legales**. Con SQL tendrias que hacer:

```sql
SELECT * FROM mensajes WHERE texto LIKE '%CONDUSEF%' OR texto LIKE '%demanda%' OR texto LIKE '%ilegal%' OR ...
```

Con ChromaDB, buscas **por significado** — y encuentra documentos relevantes aunque usen palabras distintas.

In [ ]:
# Consultas que haria el equipo de MicroPréstamos MX
queries = [
    "clientes que quieren demandar o ir a CONDUSEF",
    "personas que no pueden pagar su deuda",
    "opiniones positivas de clientes satisfechos",
    "problemas con la aplicacion o plataforma digital",
    "como es el proceso para aprobar un prestamo",
]

for query in queries:
    results = collection.query(query_texts=[query], n_results=3)

    print(f"\n🔍 Query: \"{query}\"")
    print("-" * 70)
    for i, (doc, dist, meta) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0])
    ):
        short_doc = doc[:100] + "..." if len(doc) > 100 else doc
        print(f"  {i+1}. [{meta['fuente']}] (dist: {dist:.4f}) {short_doc}")

### ¿Que observas?

- "Clientes que quieren demandar" encuentra emails de quejas Y transcripts de llamadas — **cruza canales automaticamente**
- "Personas que no pueden pagar" detecta tanto el chatbot como las llamadas de reestructura
- "Opiniones positivas" filtra los comentarios de redes sociales favorables
- Las **distancias** mas bajas = mas similares semanticamente

**Esto seria imposible con SQL `LIKE`** — tendrias que saber de antemano todas las palabras que un cliente podria usar para expresar su frustracion.

## Seccion 3 — Embeddings visuales: PCA

Los embeddings son vectores de muchas dimensiones (384+). Para verlos, los reducimos a 2D con **PCA** (Principal Component Analysis).

**Hipotesis**: documentos del mismo canal (chatbot, quejas, cobranza...) deberian quedar **cerca** en el grafico, porque hablan de temas similares.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Obtener los embeddings almacenados
all_data = collection.get(include=["embeddings", "metadatas", "documents"])
embeddings = np.array(all_data["embeddings"])
labels = [m["fuente"] for m in all_data["metadatas"]]

# Reducir a 2D con PCA
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(embeddings)

# Colores por canal
color_map = {
    "chatbot": "#2DD4BF",       # Teal
    "queja": "#FF6B6B",         # Coral
    "cobranza": "#6C5CE7",      # Purple
    "llamada": "#FFB347",       # Orange
    "redes_sociales": "#4ECDC4",# Cyan
    "interno": "#A0A0A0",       # Gray
}

plt.figure(figsize=(12, 8))
plt.style.use("dark_background")

for fuente in color_map:
    mask = [l == fuente for l in labels]
    pts = coords_2d[mask]
    if len(pts) > 0:
        plt.scatter(pts[:, 0], pts[:, 1], c=color_map[fuente], label=fuente,
                    s=120, edgecolors="white", linewidth=0.5)

# Anotar cada punto con texto recortado
for i, doc in enumerate(all_data["documents"]):
    short = doc[:30] + "..." if len(doc) > 30 else doc
    plt.annotate(short, (coords_2d[i, 0], coords_2d[i, 1]),
                 fontsize=5.5, color="white", alpha=0.7,
                 textcoords="offset points", xytext=(5, 5))

plt.title("Embeddings en 2D (PCA) — Datos de MicroPréstamos MX", fontsize=14, fontweight="bold")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
plt.legend(title="Canal", loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# Alternativa: t-SNE (mejor para separar clusters, pero no preserva distancias globales)
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=8)
coords_tsne = tsne.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
plt.style.use("dark_background")

for fuente in color_map:
    mask = [l == fuente for l in labels]
    pts = coords_tsne[mask]
    if len(pts) > 0:
        plt.scatter(pts[:, 0], pts[:, 1], c=color_map[fuente], label=fuente,
                    s=120, edgecolors="white", linewidth=0.5)

for i, doc in enumerate(all_data["documents"]):
    short = doc[:30] + "..." if len(doc) > 30 else doc
    plt.annotate(short, (coords_tsne[i, 0], coords_tsne[i, 1]),
                 fontsize=5.5, color="white", alpha=0.7,
                 textcoords="offset points", xytext=(5, 5))

plt.title("Embeddings en 2D (t-SNE) — Datos de MicroPréstamos MX", fontsize=14, fontweight="bold")
plt.legend(title="Canal", loc="best")
plt.tight_layout()
plt.show()

## Seccion 4 — Visualiza una consulta en el espacio de embeddings

Ahora vamos a ver **donde cae la pregunta de un gerente** en relacion a todos los documentos, y **lineas** hacia los resultados mas cercanos.

Ejemplo: el director de riesgos pregunta sobre clientes en problemas financieros.

In [ ]:
# Consulta del director de riesgos
query_text = "clientes que perdieron su empleo y no pueden pagar"

# Buscar y obtener el embedding de la consulta
results = collection.query(
    query_texts=[query_text],
    n_results=3,
    include=["documents", "distances", "embeddings"],
)

result_docs = results["documents"][0]
result_dists = results["distances"][0]

# Generar embedding de la query usando la misma funcion de ChromaDB
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

ef = DefaultEmbeddingFunction()
query_emb = np.array(ef([query_text]))

# Combinar con los embeddings existentes para PCA conjunto
all_embs = np.vstack([embeddings, query_emb])
pca2 = PCA(n_components=2)
all_coords = pca2.fit_transform(all_embs)

doc_coords = all_coords[:-1]
query_coord = all_coords[-1]

# Encontrar indices de los resultados
result_indices = []
for rdoc in result_docs:
    for j, d in enumerate(all_data["documents"]):
        if d == rdoc:
            result_indices.append(j)
            break

# Graficar
plt.figure(figsize=(12, 8))
plt.style.use("dark_background")

# Documentos normales (gris)
plt.scatter(doc_coords[:, 0], doc_coords[:, 1], c="#808080", s=60, alpha=0.4, label="Documentos")

# Resultados destacados
for idx in result_indices:
    plt.scatter(doc_coords[idx, 0], doc_coords[idx, 1], c="#2DD4BF", s=150,
                edgecolors="white", linewidth=1.5, zorder=5)
    plt.plot([query_coord[0], doc_coords[idx, 0]], [query_coord[1], doc_coords[idx, 1]],
             color="#2DD4BF", linewidth=1.5, alpha=0.6, linestyle="--")

# Query point
plt.scatter(query_coord[0], query_coord[1], c="#FF6B6B", s=200, marker="*",
            edgecolors="white", linewidth=1.5, zorder=10, label="Consulta del director")

# Anotar resultados
for i, idx in enumerate(result_indices):
    short = all_data["documents"][idx][:50] + "..."
    plt.annotate(f"#{i+1}: {short}", (doc_coords[idx, 0], doc_coords[idx, 1]),
                 fontsize=7, color="#2DD4BF", fontweight="bold",
                 textcoords="offset points", xytext=(8, 8))

plt.annotate(f'Query: "{query_text}"', (query_coord[0], query_coord[1]),
             fontsize=8, color="#FF6B6B", fontweight="bold",
             textcoords="offset points", xytext=(10, -15))

plt.title("Consulta del director de riesgos en el espacio de embeddings",
          fontsize=14, fontweight="bold")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

print(f"\nResultados para: \"{query_text}\"")
for i, (doc, dist) in enumerate(zip(result_docs, result_dists)):
    short = doc[:80] + "..." if len(doc) > 80 else doc
    print(f"  {i+1}. (dist: {dist:.4f}) {short}")

## Seccion 4.5 — Datos estructurados vs no estructurados

MicroPréstamos MX tambien tiene datos **estructurados** (tablas, CSVs). Vamos a cargar un dataset de llamadas de clientes para ver la diferencia.

| Tipo | Ejemplo | Herramienta |
|------|---------|-------------|
| **Estructurado** | CSV con columnas: usuario, fecha, duracion | SQL, pandas |
| **No estructurado** | "Me llaman 5 veces al dia a mi trabajo" | ChromaDB, Elasticsearch |

La magia esta en **combinar ambos**: usar SQL para filtrar por metricas y ChromaDB para entender el contexto.

In [ ]:
import pandas as pd

# Cargar dataset de llamadas (datos estructurados de la operacion)
url = "https://raw.githubusercontent.com/HesusG/course-itesm-data-mining/main/telecom_dataset_us.csv"
df = pd.read_csv(url)

print(f"📊 Dataset estructurado: {df.shape[0]} filas x {df.shape[1]} columnas\n")
print("Columnas:")
print(df.dtypes.to_string())
print(f"\n--- Primeras 5 filas ---")
df.head()

In [ ]:
# Analisis rapido con pandas (lo que SI puede hacer SQL/pandas)
print("📊 Estadisticas del dataset estructurado:\n")

print(f"Total de registros: {len(df):,}")
print(f"Usuarios unicos: {df['user_id'].nunique():,}")
print(f"Rango de fechas: {df['date'].min()} a {df['date'].max()}")
print(f"\nLlamadas perdidas: {df['is_missed_call'].sum():,} ({df['is_missed_call'].mean():.1%})")
print(f"Duracion promedio: {df['call_duration'].mean():.1f} min")

print("\n--- Top 5 operadores por volumen ---")
print(df.groupby("operator_id")["calls_count"].sum().sort_values(ascending=False).head())

print("\n" + "=" * 60)
print("💡 OBSERVA: Este analisis es perfecto para SQL/pandas.")
print("   Pero si un gerente pregunta '¿por que los clientes estan molestos?'")
print("   ... el CSV no tiene la respuesta. ChromaDB si.")

## Seccion 5 — RAG: Retrieval-Augmented Generation

RAG = **buscar contexto relevante** (ChromaDB) + **generar respuesta** (LLM).

Imagina que el equipo de atencion al cliente necesita un asistente que responda preguntas sobre politicas y casos — pero fundamentado en datos reales, no inventando respuestas.

Primero vamos a hacer RAG **sin LLM** (solo retrieval), y despues con together.ai si tienen API key.

In [ ]:
def ask_rag(question, n_results=3, api_key=None):
    """
    RAG: busca contexto en ChromaDB y genera respuesta.
    Si no hay API key, muestra solo el contexto recuperado.
    """
    # Paso 1: Retrieval — buscar documentos relevantes
    results = collection.query(query_texts=[question], n_results=n_results)
    context_docs = results["documents"][0]
    context_dists = results["distances"][0]
    context_metas = results["metadatas"][0]

    print(f"📋 Pregunta: {question}")
    print(f"\n📚 Contexto recuperado (top {n_results}):")
    for i, (doc, dist, meta) in enumerate(zip(context_docs, context_dists, context_metas)):
        short = doc[:90] + "..." if len(doc) > 90 else doc
        print(f"  {i+1}. [{meta['fuente']}] (dist: {dist:.4f}) {short}")

    # Paso 2: Generation — si hay API key, usar LLM
    if api_key:
        try:
            from together import Together

            client_llm = Together(api_key=api_key)
            context = "\n".join(f"- [{m['fuente']}] {doc}" for doc, m in zip(context_docs, context_metas))
            prompt = (
                "Eres un asistente interno de MicroPréstamos MX, una fintech mexicana de micro-créditos. "
                "Responde la pregunta usando SOLO el contexto proporcionado. "
                "Indica de que canal proviene la informacion (chatbot, queja, cobranza, etc). "
                "Si no puedes responder con el contexto, dilo.\n\n"
                f"Contexto:\n{context}\n\n"
                f"Pregunta: {question}\n\n"
                "Respuesta:"
            )

            response = client_llm.chat.completions.create(
                model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
            )
            answer = response.choices[0].message.content
            print(f"\n🤖 Respuesta del LLM:\n{answer}")
        except ImportError:
            print("\n⚠️ Instala together: !pip install together")
        except Exception as e:
            print(f"\n⚠️ Error con LLM: {e}")
    else:
        print("\n💡 Sin API key — mostrando solo retrieval.")
        print("   Para generar respuestas con LLM:")
        print('   ask_rag("tu pregunta", api_key="tu-api-key-de-together")')

    print("\n" + "=" * 60)

In [ ]:
# Probar RAG sin LLM (solo retrieval) — preguntas reales del equipo
ask_rag("¿Cuantas llamadas por dia podemos hacerle a un cliente moroso?")
ask_rag("¿Que requisitos necesita un cliente para obtener un prestamo?")
ask_rag("¿Cuales son las principales quejas de nuestros clientes?")

In [ ]:
# OPCIONAL: RAG con LLM (necesitas API key de together.ai — es gratis)
# Descomenta y reemplaza con tu API key:

# !pip install -q together
# TOGETHER_API_KEY = "tu-api-key-aqui"  # Obtener en: https://api.together.xyz
# ask_rag("Un cliente dice que le cobraron de mas y amenaza con ir a CONDUSEF. ¿Que hago?", api_key=TOGETHER_API_KEY)

## Ejercicios

Ahora te toca a ti. Intenta estos retos:

### 1. Agrega documentos de otro canal
Imagina que MicroPréstamos abre un canal de WhatsApp. Agrega 3-5 mensajes y busca si ChromaDB los encuentra con las mismas queries.

```python
collection.add(
    documents=[
        "Oye necesito pagar mi prestamo pero no tengo el numero de referencia",
        "Hola buenas noches, me aprobaron el prestamo? Ya llevo 2 dias esperando",
    ],
    metadatas=[{"fuente": "whatsapp"}, {"fuente": "whatsapp"}],
    ids=["doc_wa_1", "doc_wa_2"],
)
```

### 2. Busqueda cross-language
Intenta buscar en ingles sobre la coleccion en español. ¿Funciona?

```python
results = collection.query(query_texts=["customers who cannot pay their loans"], n_results=3)
print(results["documents"][0])
```

### 3. Filtros por canal
Busca solo en quejas formales (emails), ignorando otros canales:

```python
results = collection.query(
    query_texts=["problemas con cobros"],
    n_results=3,
    where={"fuente": "queja"},
)
for doc in results["documents"][0]:
    print(f"  - {doc[:80]}...")
```

### 4. Analisis de sentimiento manual
Busca "experiencias positivas" y "experiencias negativas". ¿ChromaDB separa bien el sentimiento?

### 5. Combina SQL + semantica
Usa pandas para encontrar usuarios con muchas llamadas perdidas, y luego busca en ChromaDB quejas relacionadas con "no me contestan":

```python
# SQL part: usuarios con mas llamadas perdidas
heavy_missed = df[df["is_missed_call"] == 1].groupby("user_id").size().sort_values(ascending=False).head(5)
print("Top 5 usuarios con llamadas perdidas:")
print(heavy_missed)

# Semantic part: ¿hay quejas sobre no poder contactar?
results = collection.query(query_texts=["no me contestan, no puedo comunicarme"], n_results=3)
print("\nQuejas relacionadas:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  [{meta['fuente']}] {doc[:80]}...")
```

## Conclusiones clave

| Concepto | Lo que aprendiste |
|----------|------------------|
| **Datos no estructurados** | Chatbots, emails, llamadas, redes sociales — el 80% de los datos empresariales |
| **ChromaDB** | Base de datos vectorial que convierte texto en embeddings y busca por significado |
| **Busqueda semantica** | Encuentra documentos relevantes sin necesidad de palabras exactas — cruza canales |
| **PCA / t-SNE** | Tecnicas para visualizar embeddings de alta dimension en 2D |
| **Datos estructurados vs no estructurados** | SQL/pandas para metricas, ChromaDB/ES para contexto y significado |
| **RAG** | Busqueda semantica + LLM = asistente que responde fundamentado en tus datos reales |

---

### Lo que acabas de construir es real

Empresas como Mercado Libre, Nu, Kavak y Clip usan exactamente estas tecnologias para:
- Analizar quejas de clientes automaticamente
- Construir chatbots que responden con informacion real (no inventada)
- Detectar patrones de fraude cruzando datos de multiples canales

**Siguiente paso**: Explora los labs de Elasticsearch en el repositorio.

📁 `github.com/HesusG/mas-alla-de-sql`